# Exercice 03 – Corrigé

Version corrigée de `03_conversation_deux_modeles.ipynb`. Exemple avec une **négociation** entre un vendeur (modèle A) et un acheteur (modèle B).

In [ ]:
import subprocess
import time

import matplotlib.pyplot as plt
import polars as pl
import requests
from openai import OpenAI

## Section 1 – Plan VRAM (exemple)

| | Modèle A | Modèle B |
|---|---|---|
| Fichier GGUF | Qwen3-8B-Q4_K_M.gguf | Llama-3.2-3B-Instruct-Q4_K_M.gguf |
| Taille du fichier | ~5,0 Go | ~2,0 Go |
| `-c` choisi | 8192 | 8192 |
| VRAM estimée | ~7 Go | ~3,5 Go |
| Port | 8080 | 8081 |

Total estimé : ~10,5 Go sur 24 Go. Large marge, nous aurions pu monter en contexte ou prendre un B plus gros.

Calcul exact du KV cache pour Qwen3-8B (36 couches, 8 têtes KV, dimension 128, f16) :
`2 × 36 × 8192 × 8 × 128 × 2 octets ≈ 1,2 Go`. Cohérent avec l'estimation.

## Section 2 – Lancer les deux serveurs

```bash
# Terminal 1
llama-server -m Qwen3-8B-Q4_K_M.gguf -c 8192 -np 1 -fa on -ngl 99 --port 8080

# Terminal 2
llama-server -m Llama-3.2-3B-Instruct-Q4_K_M.gguf -c 8192 -np 1 -fa on -ngl 99 --port 8081
```

In [ ]:
URL_A = "http://127.0.0.1:8080"
URL_B = "http://127.0.0.1:8081"

client_A = OpenAI(base_url=f"{URL_A}/v1", api_key="pas-besoin")
client_B = OpenAI(base_url=f"{URL_B}/v1", api_key="pas-besoin")

NOM_A = "Qwen3-8B-Q4"
NOM_B = "Llama3.2-3B-Q4"

for url in (URL_A, URL_B):
    print(url, requests.get(f"{url}/health").json())

# Contexte réel de chaque serveur, pour les graphiques
N_CTX = {
    NOM_A: requests.get(f"{URL_A}/props").json()["default_generation_settings"]["n_ctx"],
    NOM_B: requests.get(f"{URL_B}/props").json()["default_generation_settings"]["n_ctx"],
}
print(N_CTX)

## Section 3 – VRAM depuis Python

In [ ]:
def vram_utilisee_mb() -> int:
    sortie = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        text=True,
    )
    return int(sortie.strip().splitlines()[0])   # première ligne = premier GPU

print(vram_utilisee_mb(), "Mo utilisés avec les deux serveurs chargés")

## Section 4 – La conversation

In [ ]:
SYSTEM_A = """Tu es Marc, vendeur d'un vélo de course d'occasion en très bon état.
Ton prix affiché est 800 euros. Tu ne descendras jamais sous 600 euros, mais tu ne le dis pas.
Tu es sympathique mais tu défends ton prix avec des arguments concrets.
Réponds en 2 à 4 phrases, en français, sans didascalies."""

SYSTEM_B = """Tu es Sophie, tu veux acheter un vélo de course d'occasion.
Ton budget maximum est 650 euros, mais tu ne le révèles pas. Tu veux payer le moins possible.
Tu poses des questions sur l'état du vélo et tu négocies fermement.
Quand un accord est trouvé, dis explicitement « Marché conclu ».
Réponds en 2 à 4 phrases, en français, sans didascalies."""

AMORCE = "Bonjour ! Je viens pour le vélo de course que vous vendez. Il est toujours disponible ?"
N_TOURS = 10


def tour(client, history: list, message: str) -> tuple[str, dict]:
    history.append({"role": "user", "content": message})
    debut = time.perf_counter()
    r = client.chat.completions.create(
        model="local",
        messages=history,
        temperature=0.7,
        max_tokens=250,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    reponse = r.choices[0].message.content.strip()
    history.append({"role": "assistant", "content": reponse})

    t = r.model_extra["timings"]
    metriques = {
        "prompt_tokens": r.usage.prompt_tokens,
        "completion_tokens": r.usage.completion_tokens,
        "prompt_n": t["prompt_n"],
        "tok_par_s": round(t["predicted_per_second"], 1),
        "duree_ms": round((time.perf_counter() - debut) * 1000),
        "finish_reason": r.choices[0].finish_reason,
    }
    return reponse, metriques


history_A = [{"role": "system", "content": SYSTEM_A}]
history_B = [{"role": "system", "content": SYSTEM_B}]
journal = []

# L'amorce est le premier message que A (le vendeur) reçoit : c'est B (l'acheteuse) qui "parle" en premier.
# Nous l'ajoutons donc aussi dans l'historique de B comme si elle l'avait dit.
history_B.append({"role": "assistant", "content": AMORCE})
print(f"[{NOM_B}] {AMORCE}\n")

message_pour_A = AMORCE

for n in range(1, N_TOURS + 1):
    # --- A parle ---
    reponse_A, m = tour(client_A, history_A, message_pour_A)
    journal.append({"tour": n, "modele": NOM_A, **m, "vram_mb": vram_utilisee_mb(), "texte": reponse_A})
    print(f"[{NOM_A}] {reponse_A}\n")

    # --- B parle : ce que A a dit devient un message 'user' pour B ---
    reponse_B, m = tour(client_B, history_B, reponse_A)
    journal.append({"tour": n, "modele": NOM_B, **m, "vram_mb": vram_utilisee_mb(), "texte": reponse_B})
    print(f"[{NOM_B}] {reponse_B}\n")

    if "marché conclu" in reponse_B.lower():
        print(f"Accord trouvé au tour {n}.")
        break

    message_pour_A = reponse_B

print(f"{len(journal)} messages enregistrés.")

## Section 5 – Sauvegarde et graphiques

In [ ]:
df = pl.DataFrame(journal)
df.write_csv("conversation.csv")
df.drop("texte")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Croissance du contexte : le pivot donne une colonne par modèle
large = df.pivot(on="modele", index="tour", values="prompt_tokens").sort("tour")
for nom in (NOM_A, NOM_B):
    axes[0, 0].plot(large["tour"], large[nom], marker="o", label=nom)
for nom, ctx in N_CTX.items():
    axes[0, 0].axhline(ctx, linestyle="--", alpha=0.5, label=f"limite {nom}")
axes[0, 0].set_title("Tokens du prompt par tour")
axes[0, 0].legend()

# 2. Longueur des réponses
large = df.pivot(on="modele", index="tour", values="completion_tokens").sort("tour")
for nom in (NOM_A, NOM_B):
    axes[0, 1].plot(large["tour"], large[nom], marker="o", label=nom)
axes[0, 1].set_title("Tokens générés par tour")
axes[0, 1].legend()

# 3. Vitesse
vitesse = df.group_by("modele").agg(pl.col("tok_par_s").mean()).sort("modele")
axes[1, 0].bar(vitesse["modele"], vitesse["tok_par_s"])
axes[1, 0].set_title("Vitesse moyenne (tok/s)")

# 4. Effet du cache : prompt_tokens vs prompt_n pour le modèle A
sous = df.filter(pl.col("modele") == NOM_A).sort("tour")
axes[1, 1].plot(sous["tour"], sous["prompt_tokens"], marker="o", label="prompt_tokens")
axes[1, 1].plot(sous["tour"], sous["prompt_n"], marker="o", label="prompt_n")
axes[1, 1].set_title(f"{NOM_A} : tokens envoyés vs réellement traités")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Chiffres pour l'analyse
print("Tokens économisés par le cache :")
print(df.group_by("modele").agg(
    (pl.col("prompt_tokens") - pl.col("prompt_n")).sum().alias("tokens_economises")
).sort("modele"))

print("\nVitesse et longueur moyennes :")
print(df.group_by("modele").agg(
    pl.col("tok_par_s", "completion_tokens", "duree_ms").mean().round(1)
).sort("modele"))

print("\nProjection : à ce rythme, tours avant d'atteindre la limite de contexte :")
for nom in (NOM_A, NOM_B):
    s = df.filter(pl.col("modele") == nom).sort("tour")
    pt = s["prompt_tokens"]
    croissance = (pt[-1] - pt[0]) / max(len(s) - 1, 1)
    restant = N_CTX[nom] - pt[-1]
    print(f"  {nom}: +{croissance:.0f} tokens/tour → ~{restant / croissance:.0f} tours restants")

## Section 6 – Éléments d'analyse

1. **Premier à saturer** : les deux historiques contiennent le même dialogue, donc à peu près le même nombre de mots. Les différences viennent du **tokenizer** (un modèle peut compter 10 à 20 % de tokens en plus sur le même texte français), du **system prompt** (longueurs différentes) et de la taille de `-c`. La projection ci-dessus le chiffre.
2. **Objectif atteint ?** Avec des objectifs contradictoires (600 minimum vs 650 maximum), un accord entre 600 et 650 est possible. Souvent, l'un des modèles cède trop vite, ou les deux se répètent à partir du tour 6-7. Regardez si `completion_tokens` diminue : c'est le signe de réponses de plus en plus courtes et formelles.
3. **Cache** : `prompt_n` reste proche de la longueur du dernier message reçu (quelques dizaines de tokens), alors que `prompt_tokens` grimpe à plusieurs centaines ou milliers. Le total économisé est affiché ci-dessus.
4. **Vitesse** : le 3B devrait être nettement plus rapide que le 8B (moins de poids à lire en mémoire à chaque token). Les deux serveurs ne génèrent jamais en même temps dans cette boucle (l'un attend l'autre), donc le partage du GPU n'a quasiment pas d'effet ici. Ce serait différent avec des appels concurrents.
5. **Plan VRAM avec le recul** : ~10 Go utilisés sur 24, nous aurions pu prendre un modèle B plus gros (8B aussi) ou augmenter `-c` à 16k-32k pour une conversation beaucoup plus longue.